# CFRM 521/421 Machine Learning for Finance
## Course Project Template

**Project Title:** Volatility Prediction of Microsoft Using Machine Learning

**Group Members:**  
- Jenny Fu (email: jennyfu1@uw.edu, Algorithm: Support Vector Regression)
- Emma Xu (email: zihanxu@uw.edu, Algorithm: Gradient Boosing)  
- Shawn Wen (email: ___, Algorithm: Neural Network with MLP)  
- Summer Zhu (email: ___, Algorithm: Random Forest)   

**Date:**

<span style="color:red">
**Please remove the bullet points in each section as you proceed and feel free to adjust the structure and contents as needed.
</span>

# 1. Introduction

## 1.1 Problem Statement
- What is the goal of your project?
- Why is this problem important in finance?
- What financial decision, prediction, or classification task are you studying?

## 1.2 Related Literature
- Briefly summarize relevant papers or prior work.
- Explain how your project relates to existing studies.
- Cite all sources properly.

## 1.3 Contribution
- What does your project add beyond existing work?
- Is your contribution empirical comparison, replication with extensions, new data, or a new application?


# 2. Data Description

## 2.1 Data Source
- Describe the original source of the data.
- Include links, API names, or repository names if relevant.

## 2.2 Data Structure
- What does each row represent?
- What is the sampling frequency?
- How many observations are there?
- What time period does the data cover?

## 2.3 Target Variable
- Define the response variable clearly.

## 2.4 Features
- List and briefly describe the predictors.
- Give a few examples.

## 2.5 Data Cleaning and Preprocessing
- Missing values
- Outlier handling
- Scaling or normalization
- Feature engineering
- Train/validation/test split


In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reproducibility
np.random.seed(42)


In [16]:
import yfinance as yf

# Load data
tickers = ['MSFT', 'SPY', 'QQQ', 'XLK', '^VIX']

data = yf.download(
    tickers,
    start='2016-01-01',
    end='2026-01-01',
    auto_adjust=True
)

data.head()


[*********************100%***********************]  5 of 5 completed


Price           Close                                                \
Ticker           MSFT         QQQ         SPY        XLK       ^VIX   
Date                                                                  
2016-01-04  47.770710  101.717285  169.908157  18.860455  20.700001   
2016-01-05  47.988655  101.540779  170.195496  18.811369  19.340000   
2016-01-06  47.116924  100.565430  168.048630  18.579351  20.590000   
2016-01-07  45.478069   97.416351  164.016891  18.030539  24.990000   
2016-01-08  45.617554   96.617493  162.216568  17.887753  27.010000   

Price            High                                                ...  \
Ticker           MSFT         QQQ         SPY        XLK       ^VIX  ...   
Date                                                                 ...   
2016-01-04  47.770710  101.810176  169.916605  18.864916  23.360001  ...   
2016-01-05  48.285043  102.348946  170.651914  18.963074  21.059999  ...   
2016-01-06  47.422031  101.150650  169.096707  18.704284  21.860001  ...   
2016-01-07  46.628754   99.664348  166.882221  18.436571  25.860001  ...   
2016-01-08  46.445693   98.735441  165.538329  18.249167  27.080000  ...   

Price            Open                                                  Volume  \
Ticker           MSFT         QQQ         SPY        XLK       ^VIX      MSFT   
Date                                                                            
2016-01-04  47.352281  101.670836  169.460186  18.766755  22.480000  53778000   
2016-01-05  47.884049  102.218897  170.229299  18.918455  20.750000  34079700   
2016-01-06  47.352291   99.775845  167.642910  18.534733  21.670000  39518900   
2016-01-07  45.940088   98.419585  165.098785  18.200091  23.219999  56564900   
2016-01-08  45.652420   98.122347  164.980474  18.186699  22.959999  48754000   

Price                                           
Ticker           QQQ        SPY       XLK ^VIX  
Date                                            
2016-01-04  50807600  222353500  43277200    0  
2016-01-05  38795200  110845800  32134400    0  
2016-01-06  41891100  152112600  27716800    0  
2016-01-07  61386300  213436100  33681400    0  
2016-01-08  69344000  209817200  38466200    0  

[5 rows x 25 columns]

In [17]:
# Flatten MultiIndex columns
data.columns = [f"{ticker}_{price}" for price, ticker in data.columns]

# Rename ^VIX to VIX for easier use
data = data.rename(columns=lambda x: x.replace('^VIX', 'VIX'))

# Check column names
data.columns

# Create a working dataframe
df = data.copy()

# =========================
# 1. Daily returns
# =========================

df['MSFT_return_1d'] = df['MSFT_Close'].pct_change()
df['SPY_return_1d'] = df['SPY_Close'].pct_change()
df['QQQ_return_1d'] = df['QQQ_Close'].pct_change()
df['XLK_return_1d'] = df['XLK_Close'].pct_change()


# =========================
# 2. MSFT lag returns
# =========================

df['MSFT_return_lag1'] = df['MSFT_return_1d'].shift(1)
df['MSFT_return_lag3'] = df['MSFT_return_1d'].shift(3)
df['MSFT_return_lag5'] = df['MSFT_return_1d'].shift(5)
df['MSFT_return_lag10'] = df['MSFT_return_1d'].shift(10)
df['MSFT_return_lag15'] = df['MSFT_return_1d'].shift(15)
df['MSFT_return_lag21'] = df['MSFT_return_1d'].shift(21)


# =========================
# 3. MSFT rolling volatility
# =========================

df['MSFT_rolling_vol_5'] = df['MSFT_return_1d'].rolling(window=5).std()
df['MSFT_rolling_vol_21'] = df['MSFT_return_1d'].rolling(window=21).std()


# =========================
# 4. MSFT log volume
# =========================

df['MSFT_log_volume'] = np.log(df['MSFT_Volume'])


# =========================
# 5. SPY market-wide features
# =========================

df['SPY_return_lag1'] = df['SPY_return_1d'].shift(1)
df['SPY_rolling_vol_5'] = df['SPY_return_1d'].rolling(window=5).std()
df['SPY_rolling_vol_21'] = df['SPY_return_1d'].rolling(window=21).std()


# =========================
# 6. QQQ technology market features
# =========================

df['QQQ_rolling_vol_21'] = df['QQQ_return_1d'].rolling(window=21).std()


# =========================
# 7. VIX features
# =========================

df['VIX_level'] = df['VIX_Close']
df['VIX_change'] = df['VIX_Close'].pct_change()
df['VIX_lag1'] = df['VIX_Close'].shift(1)


# =========================
# 8. XLK sector ETF features
# =========================

df['XLK_rolling_vol_5'] = df['XLK_return_1d'].rolling(window=5).std()
df['XLK_rolling_vol_21'] = df['XLK_return_1d'].rolling(window=21).std()


# =========================
# 9. Relative performance features
# =========================

df['MSFT_minus_XLK_return'] = df['MSFT_return_1d'] - df['XLK_return_1d']
df['MSFT_minus_SPY_return'] = df['MSFT_return_1d'] - df['SPY_return_1d']


# Check result
df.head()

# =========================
# 10. Target variables
# =========================

# Daily prediction target
df['target_volatility_1d'] = df['MSFT_return_1d'].shift(-1).abs()

# Weekly prediction target
df['target_volatility_5d'] = (
    df['MSFT_return_1d']
    .shift(-1)
    .rolling(window=5)
    .std()
    .shift(-4)
)

# Monthly prediction target
df['target_volatility_21d'] = (
    df['MSFT_return_1d']
    .shift(-1)
    .rolling(window=21)
    .std()
    .shift(-20)
)

In [18]:
# =========================
# 11. Final model dataset
# =========================

feature_cols = [
    'MSFT_return_1d',
    'MSFT_return_lag1',
    'MSFT_return_lag3',
    'MSFT_return_lag5',
    'MSFT_return_lag10',
    'MSFT_return_lag15',
    'MSFT_return_lag21',
    'MSFT_rolling_vol_5',
    'MSFT_rolling_vol_21',
    'MSFT_log_volume',
    
    'SPY_return_1d',
    'SPY_return_lag1',
    'SPY_rolling_vol_5',
    'SPY_rolling_vol_21',
    
    'QQQ_return_1d',
    'QQQ_rolling_vol_21',
    
    'VIX_level',
    'VIX_change',
    'VIX_lag1',
    
    'XLK_return_1d',
    'XLK_rolling_vol_5',
    'XLK_rolling_vol_21',
    
    'MSFT_minus_XLK_return',
    'MSFT_minus_SPY_return'
]

target_cols = [
    'target_volatility_1d',
    'target_volatility_5d',
    'target_volatility_21d'
]

# Dataset before dropna
model_data_before_dropna = df[feature_cols + target_cols].copy()

# Total number of NaN values
total_na = model_data_before_dropna.isna().sum().sum()
print("Shape before dropna:", model_data_before_dropna.shape)
print("Total NaN values before dropna:", total_na)

# NaN values by column
na_by_column = model_data_before_dropna.isna().sum()
na_by_column[na_by_column > 0]
print("NaN values by column:", na_by_column[na_by_column > 0])

# Number of rows with at least one NaN
rows_with_na = model_data_before_dropna.isna().any(axis=1).sum()
print("Rows with at least one NaN:", rows_with_na)

# Keep only features and targets, then drop missing values
model_data = df[feature_cols + target_cols].dropna()

print("Final model_data shape:", model_data.shape)

model_data.head()

Shape before dropna: (2514, 27)
Total NaN values before dropna: 197
NaN values by column: MSFT_return_1d            1
MSFT_return_lag1          2
MSFT_return_lag3          4
MSFT_return_lag5          6
MSFT_return_lag10        11
MSFT_return_lag15        16
MSFT_return_lag21        22
MSFT_rolling_vol_5        5
MSFT_rolling_vol_21      21
SPY_return_1d             1
SPY_return_lag1           2
SPY_rolling_vol_5         5
SPY_rolling_vol_21       21
QQQ_return_1d             1
QQQ_rolling_vol_21       21
VIX_change                1
VIX_lag1                  1
XLK_return_1d             1
XLK_rolling_vol_5         5
XLK_rolling_vol_21       21
MSFT_minus_XLK_return     1
MSFT_minus_SPY_return     1
target_volatility_1d      1
target_volatility_5d      5
target_volatility_21d    21
dtype: int64
Rows with at least one NaN: 43
Final model_data shape: (2471, 27)


,MSFT_return_1d,MSFT_return_lag1,MSFT_return_lag3,MSFT_return_lag5,MSFT_return_lag10,MSFT_return_lag15,MSFT_return_lag21,MSFT_rolling_vol_5,MSFT_rolling_vol_21,MSFT_log_volume,...,VIX_change,VIX_lag1,XLK_return_1d,XLK_rolling_vol_5,XLK_rolling_vol_21,MSFT_minus_XLK_return,MSFT_minus_SPY_return,target_volatility_1d,target_volatility_5d,target_volatility_21d
Date,,,,,,,,,,,,,,,,,,,,,
2016-02-04,-0.003067,-0.015849,-0.006898,0.016400,-0.006104,-0.021599,0.004562,0.034177,0.023677,17.665384,...,0.008776,21.650000,0.000985,0.017823,0.017475,-0.004052,-0.004635,0.035385,0.017031,0.017589
2016-02-05,-0.035385,-0.003067,-0.031256,0.058202,0.035856,0.028466,-0.018165,0.014384,0.024531,17.942790,...,0.070513,21.840000,-0.028052,0.013787,0.018276,-0.007333,-0.016334,0.014952,0.011860,0.015833
2016-02-08,-0.014952,-0.035385,-0.015849,-0.006898,-0.009562,-0.039917,-0.034783,0.013160,0.023622,17.897960,...,0.112062,23.379999,-0.014430,0.012069,0.017430,-0.000522,-0.001491,0.002631,0.009653,0.015987
2016-02-09,-0.002631,-0.014952,-0.003067,-0.031256,0.007337,-0.008433,0.003067,0.013320,0.023590,17.660122,...,0.020769,26.000000,-0.004623,0.011943,0.017385,0.001992,-0.002686,0.008726,0.010121,0.016449
2016-02-10,0.008726,-0.002631,-0.035385,-0.015849,-0.018210,0.004549,-0.000573,0.016737,0.023716,17.459314,...,-0.009420,26.540001,0.001548,0.012475,0.017305,0.007177,0.009589,0.000402,0.013073,0.016803


In [19]:
# =========================
# 14. Time-series train / validation / test split
# =========================

# Make sure data is sorted by date
model_data = model_data.sort_index()

# Split indices
n = len(model_data)

train_end = int(n * 0.70)
valid_end = int(n * 0.85)

# Split data
train_data = model_data.iloc[:train_end]
valid_data = model_data.iloc[train_end:valid_end]
test_data = model_data.iloc[valid_end:]

print("Total observations:", n)
print("Train shape:", train_data.shape)
print("Validation shape:", valid_data.shape)
print("Test shape:", test_data.shape)

print("Train period:", train_data.index.min(), "to", train_data.index.max())
print("Validation period:", valid_data.index.min(), "to", valid_data.index.max())
print("Test period:", test_data.index.min(), "to", test_data.index.max())

# =========================
# 15. Define X variables
# =========================

X_train = train_data[feature_cols]
X_valid = valid_data[feature_cols]
X_test = test_data[feature_cols]


# =========================
# 16. Define y variables for each prediction horizon
# =========================

# Daily target
y_train_1d = train_data['target_volatility_1d']
y_valid_1d = valid_data['target_volatility_1d']
y_test_1d = test_data['target_volatility_1d']

# Weekly target
y_train_5d = train_data['target_volatility_5d']
y_valid_5d = valid_data['target_volatility_5d']
y_test_5d = test_data['target_volatility_5d']

# Monthly target
y_train_21d = train_data['target_volatility_21d']
y_valid_21d = valid_data['target_volatility_21d']
y_test_21d = test_data['target_volatility_21d']

Total observations: 2471
Train shape: (1729, 27)
Validation shape: (371, 27)
Test shape: (371, 27)
Train period: 2016-02-04 00:00:00 to 2022-12-14 00:00:00
Validation period: 2022-12-15 00:00:00 to 2024-06-07 00:00:00
Test period: 2024-06-10 00:00:00 to 2025-12-01 00:00:00


In [20]:
# =========================
# 17. Standardize X variables
# =========================

from sklearn.preprocessing import StandardScaler
import pandas as pd

# Create scaler
scaler = StandardScaler()

# Fit scaler only on training data
scaler.fit(X_train)

# Transform train, validation, and test using the same scaler
X_train_scaled = scaler.transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

# Convert scaled arrays back to DataFrames
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    index=X_train.index,
    columns=feature_cols
)

X_valid_scaled = pd.DataFrame(
    X_valid_scaled,
    index=X_valid.index,
    columns=feature_cols
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    index=X_test.index,
    columns=feature_cols
)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_valid_scaled shape:", X_valid_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

X_train_scaled shape: (1729, 24)
X_valid_scaled shape: (371, 24)
X_test_scaled shape: (371, 24)


# 3. Exploratory Data Analysis

- Summarize the data.
- Visualize key variables.
- Examine the target variable.
- Discuss stylized facts or empirical patterns relevant to the financial problem.


In [ ]:
# Basic summary statistics
# df.describe()


In [ ]:
# Example visualization
# plt.figure(figsize=(6,4))
# df['your_column'].hist(bins=30)
# plt.title('Distribution of your_column')
# plt.show()


# 4. Methodology

Clearly separate the problem description from the learning algorithms.

Include a separate subsection for hyperparameter tuning:
- Explain how tuning is performed.
- Make the comparison fair across models.
- State the validation procedure clearly.

## 4.1 Overview of Models
- You must try at least as many algorithms as group members.
- Each group member should implement at least one algorithm.
- Use models within the scope of the course.
- If using a more advanced model, provide sufficient background and compare it against standard baselines first.


## 4.2 Model 1: ___

**Implemented by:** 

- Motivation
- Model description
- Why this method is appropriate/Key assumptions


## 4.3 Model 2: ___

**Implemented by:** 

- Motivation
- Model description
- Why this method is appropriate


Repeat until Model N (N=your group size)


# 5. Results

Clearly separate the presentation of results from the conclusions.

## 5.1 Evaluation Metrics
- Explain why the chosen metrics are appropriate.

## 5.2 Main Quantitative Results
- Present results in tables. Compare model performance after tuning.


## 5.3 Visualizations
- Prediction vs actual
- Residual plots
- Feature importance
- Confusion matrix or ROC curve if classification


# 6. Discussions and Conclusions

Make it brief; (2-3 paragraphs max)

Discuss:
- Which model performed best?
- Why do you think it performed best?
- What do the results mean in the financial context?
- Are there economic or practical implications?
- What are the limitations of the study?

Conclude:
- Summarize the main findings.
- State the major takeaway.
- Suggest possible future work.

# Appendix. Reproducibility

- State the software environment.
- State package versions if relevant.
- Explain how to reproduce the analysis.
- Ensure the notebook has been run from start to finish.


In [ ]:
# example: package versions
# import sys
# print(sys.version)
# print(pd.__version__)
# print(np.__version__)

# References

- Include all papers, datasets, websites, and software packages cited in the notebook.
- Use a consistent citation style.
